# Qwen3.8 Threshold MoE — Colab Runner
Runner only: clone, install, mount Drive, inspect, validate, probe data, train, resume, and plot. Model/training implementation stays in GitHub.

In [ ]:
!rm -rf /content/My-works
!git clone -b moe-threshold-pretrain https://github.com/Logan17de/My-works.git

In [ ]:
%cd /content/My-works/qwen38-threshold-moe
!pip install -r requirements.txt

## Mount Google Drive
The `--drive-root` CLI below makes both checkpoints/logs and the mmap expert masters persistent across Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Inspect effective setup
This confirms paths and the exact CLI overrides before allocating expert files.

In [ ]:
!python train.py inspect --config configs/colab_smoke.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --save-every 1 --plot-every 1

In [ ]:
!python train.py validate-layerwise --config configs/colab_smoke.json

In [ ]:
!python train.py probe-data --config configs/colab_smoke.json --samples 2

## Smoke train — persistent in Drive
Change important experiment knobs directly in the command; no JSON editing is required.

In [ ]:
!python train.py train --config configs/colab_smoke.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name colab-smoke --save-every 1 --plot-every 1 --max-steps 5 --cache-layers 2

## Resume from latest checkpoint
This continues the same Drive-backed expert store and optimizer state.

In [ ]:
!python train.py train --config configs/colab_smoke.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name colab-smoke --resume latest --save-every 1 --max-steps 10 --cache-layers 2

## Full 80-layer / 500-expert run
The Drive-backed BF16 routed-expert masters alone are about 117 GiB, so verify Drive capacity first. `--cache-layers` controls the sliding GPU expert window.

In [ ]:
!python train.py inspect --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name full-g4 --save-every 10 --cache-layers 5 --max-steps 1000

In [ ]:
!python train.py train --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name full-g4 --save-every 10 --plot-every 5 --cache-layers 5 --max-steps 1000

## Resume full run

In [ ]:
!python train.py train --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name full-g4 --resume latest --save-every 10 --plot-every 5 --cache-layers 5 --max-steps 1000

## Rebuild graphs from CSV

In [ ]:
!python train.py plot --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --run-name full-g4